# 07 — Tourism Intensity

Quantifies tourism presence per census tract from OSM tourism-related POIs.

**Data source:** Overpass API (OSM `tourism=*`) — fully portable.

**Output columns:** `tract_id`, `hotel_count`, `tourism_poi_count`, `tourism_density`, `tourism_ratio`

**Output file:** `csv/07_tourism_intensity.csv`

In [ ]:
ZONES_CONFIG = "zones.json"
QUERY_RADIUS = 500

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import json
import os
import math
import hashlib

os.makedirs("csv", exist_ok=True)
os.makedirs("cache", exist_ok=True)

df_tracts = pd.read_csv("csv/01_zone_definition.csv", dtype={"tract_id": str})
print(f"Loaded {len(df_tracts)} tracts")

# Load amenity totals from notebook 02 for ratio computation
df_amenities = pd.read_csv("csv/02_amenity_composition.csv", dtype={"tract_id": str})
amenity_totals = dict(zip(df_amenities["tract_id"], df_amenities["amenity_count_total"]))

In [ ]:
OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
HEADERS = {"User-Agent": "zone-finding/1.0 (research project)"}

HOTEL_VALUES = {"hotel", "hostel", "motel", "guest_house"}
TOURISM_VALUES = {"hotel", "hostel", "motel", "guest_house", "museum",
                  "attraction", "viewpoint", "gallery", "artwork",
                  "information", "theme_park", "zoo", "aquarium"}


def _cache_path(query):
    h = hashlib.sha1(query.encode()).hexdigest()
    return f"cache/{h}.json"


def query_overpass_cached(query, max_retries=3):
    cp = _cache_path(query)
    if os.path.exists(cp):
        with open(cp, encoding="utf-8") as f:
            return json.load(f)
    last_error = None
    for attempt in range(max_retries):
        ep = OVERPASS_ENDPOINTS[attempt % len(OVERPASS_ENDPOINTS)]
        try:
            r = requests.post(ep, data={"data": query}, headers=HEADERS, timeout=90)
            r.raise_for_status()
            data = r.json()
            with open(cp, "w", encoding="utf-8") as f:
                json.dump(data, f)
            return data
        except Exception as e:
            last_error = e
            time.sleep(3 + attempt * 2)
    raise RuntimeError(f"Overpass failed: {last_error}")


print("Helpers ready.")

In [ ]:
# ── Query tourism POIs per tract ──────────────────────

AREA_KM2 = math.pi * (QUERY_RADIUS / 1000) ** 2
records = []
n_tracts = len(df_tracts)

for i, row in df_tracts.iterrows():
    tract_id = row["tract_id"]
    lat, lon = row["tract_lat"], row["tract_lon"]
    
    if (i + 1) % 25 == 0 or i == 0:
        print(f"  [{i+1}/{n_tracts}] Tract {tract_id}")
    
    query = (f'[out:json][timeout:30];\n'
             f'(node["tourism"](around:{QUERY_RADIUS},{lat},{lon});\n'
             f' way["tourism"](around:{QUERY_RADIUS},{lat},{lon}););\n'
             f'out center tags;')
    
    try:
        data = query_overpass_cached(query)
    except Exception as e:
        print(f"  ERROR tract {tract_id}: {e}")
        records.append({"tract_id": tract_id, "hotel_count": 0,
                        "tourism_poi_count": 0, "tourism_density": 0.0, "tourism_ratio": 0.0})
        continue
    
    hotel_count = 0
    tourism_count = 0
    
    for el in data.get("elements", []):
        tags = el.get("tags", {})
        tourism_val = tags.get("tourism", "")
        if tourism_val in TOURISM_VALUES:
            tourism_count += 1
        if tourism_val in HOTEL_VALUES:
            hotel_count += 1
    
    total_amenities = amenity_totals.get(tract_id, 0)
    tourism_ratio = round(tourism_count / total_amenities, 4) if total_amenities > 0 else 0.0
    
    records.append({
        "tract_id": tract_id,
        "hotel_count": hotel_count,
        "tourism_poi_count": tourism_count,
        "tourism_density": round(tourism_count / AREA_KM2, 2),
        "tourism_ratio": tourism_ratio,
    })
    
    time.sleep(0.5)

df_tourism = pd.DataFrame(records)
print(f"\nCompleted: {len(df_tourism)} tracts")
print(f"Tracts with hotels: {(df_tourism['hotel_count'] > 0).sum()}")
print(f"Tracts with tourism POIs: {(df_tourism['tourism_poi_count'] > 0).sum()}")

In [ ]:
# ── Save output ───────────────────────────────────────
output_path = "csv/07_tourism_intensity.csv"
df_tourism.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_tourism)} rows x {df_tourism.shape[1]} cols)")
print(df_tourism.describe().round(2).to_string())